1. Instalasi & Import Library

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re
from datetime import datetime, timedelta


2. Load & Merge Dataset Mentah

In [2]:
# Menggunakan '../' karena notebook ini berada di folder 'notebooks'
path_raw = '../data/raw_x/'
all_files = glob.glob(os.path.join(path_raw, "*.csv"))

df_list = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        
        # Ekstrak nama file (tanpa .csv) untuk tahu dari file mana data ini berasal
        kategori_asal = os.path.basename(file).replace('.csv', '')
        df_temp['kategori_sumber'] = kategori_asal
        
        df_list.append(df_temp)
    except Exception as e:
        print(f"Gagal membaca file {file}: {e}")

# Gabungkan semua DataFrame menjadi satu
df_raw = pd.concat(df_list, ignore_index=True)

print(f"Total dataset mentah yang berhasil di-load: {df_raw.shape[0]} baris.")
display(df_raw.head(3))

Total dataset mentah yang berhasil di-load: 1873 baris.


,web_scraper_order,web_scraper_start_url,data,data2,data3,data4,data5,data6,data8,kategori_sumber,data7,data9,data10,data12,data11,data13
0,1775290311-1,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,8m,Koh Alung,@alung_koh21471,krisis air,Pemerintah berupaya menangani,bersih di Karang Baru secara bertahap.,3,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN
1,1775290311-2,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,16m,Batampos,@BatamPos,"hingga Hari ke -70, Warga Tanjung Sengkuang Ma...",Krisis Air,NaN,5,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN
2,1775290311-3,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,17m,Rizki Fadillnaldo,@ciroalves2055,KRISIS AIR,NaN,BERSIH? JANGAN ASAL MENYALAHKAN!\nMasalahnya l...,3,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN


3. Menyatukan Teks yang Terpecah

In [3]:
print("Memulai proses pembersihan dan penyatuan teks...")

# Identifikasi kolom waktu (biasanya bernama 'data')
time_col = 'data' if 'data' in df_raw.columns else None

# Ambil semua kolom teks hasil scraper (berawalan 'data' tapi bukan kolom waktu)
text_cols = [c for c in df_raw.columns if c.startswith('data') and c != time_col]

def bersihkan_dan_gabung(row):
    teks_gabungan = []
    username = 'Tidak diketahui'
    
    for col in text_cols:
        val = str(row[col]) if pd.notna(row[col]) else ''
        val = val.strip()
        if not val:
            continue
            
        # Jika kata berawalan @ dan berdiri sendiri, jadikan username
        if val.startswith('@') and len(val.split()) == 1:
            username = val
        # Jika bukan spam angka (seperti jumlah like/retweet), masukkan ke teks
        elif not (val.isdigit() and len(val) < 4):
            teks_gabungan.append(val)
            
    return pd.Series([username, ' '.join(teks_gabungan)])

# Terapkan fungsi ke seluruh baris
df_raw[['username', 'teks_bersih']] = df_raw.apply(bersihkan_dan_gabung, axis=1)

# Buat dataframe baru untuk data interim
df_clean = df_raw.copy()

# Pindahkan kolom waktu
if time_col:
    df_clean['waktu'] = df_clean[time_col]
else:
    df_clean['waktu'] = 'Tidak diketahui'

# Buang baris yang teksnya kosong setelah dibersihkan
df_clean = df_clean[df_clean['teks_bersih'] != '']

print(f"Total baris setelah teks disatukan & dibersihkan: {df_clean.shape[0]}")
display(df_clean[['waktu', 'username', 'teks_bersih']].head(3))

Memulai proses pembersihan dan penyatuan teks...
Total baris setelah teks disatukan & dibersihkan: 1863


,waktu,username,teks_bersih
0,8m,@alung_koh21471,Koh Alung krisis air Pemerintah berupaya menan...
1,16m,@BatamPos,"Batampos hingga Hari ke -70, Warga Tanjung Sen..."
2,17m,@ciroalves2055,Rizki Fadillnaldo KRISIS AIR BERSIH? JANGAN AS...


4. Restrukturisasi Kolom Target ADUIN

In [4]:
print("Menyusun arsitektur kolom...")

# 1. Jadikan asal file sebagai kolom kategori utama
df_clean['kategori_isu'] = df_clean['kategori_sumber']

# 2. Setup Kolom Target NLP dengan nilai Placeholder
df_clean['urgensi'] = 'Belum Dilabeli'
df_clean['sentimen'] = 'Belum Dilabeli'
df_clean['lokasi_insiden'] = 'Tidak diketahui'

# 3. Restrukturisasi Urutan Kolom Akhir
kolom_urut = ['waktu', 'username', 'teks_bersih', 'kategori_isu', 'urgensi', 'sentimen', 'lokasi_insiden']
df_aduin = df_clean[kolom_urut].copy()

# Hapus duplikat berdasarkan teks keluhan agar data benar-benar unik
df_aduin = df_aduin.drop_duplicates(subset=['username', 'waktu', 'teks_bersih']).reset_index(drop=True)

print("Blueprint tabel ADUIN berhasil diperbarui!")
display(df_aduin.head(3))

# Cek kembali
print("\n--- Pengecekan Distribusi Kategori ---")
print(df_aduin['kategori_isu'].value_counts())

Menyusun arsitektur kolom...
Blueprint tabel ADUIN berhasil diperbarui!


,waktu,username,teks_bersih,kategori_isu,urgensi,sentimen,lokasi_insiden
0,8m,@alung_koh21471,Koh Alung krisis air Pemerintah berupaya menan...,Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1,16m,@BatamPos,"Batampos hingga Hari ke -70, Warga Tanjung Sen...",Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
2,17m,@ciroalves2055,Rizki Fadillnaldo KRISIS AIR BERSIH? JANGAN AS...,Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui



--- Pengecekan Distribusi Kategori ---
kategori_isu
Infrastruktur & Jalan    416
Ekonomi & UMKM           388
Lingkungan & Sampah      176
Pelayanan Publik         163
Pendidikan               156
Air & Sanitasi           145
Keamanan & Sosial        128
Transportasi             106
Bencana & Cuaca          103
Kesehatan                 37
Name: count, dtype: int64


5. Data Cleaning

5.1. Drop Baris Noise & Tidak Relevan

In [5]:
df_clean = df_aduin.copy()
n_awal = len(df_clean)

# Teks bahasa Thai
mask_thai = df_clean['teks_bersih'].str.contains(r'[\u0E00-\u0E7F]', regex=True, na=False)
df_clean = df_clean[~mask_thai]
print(f'Drop bahasa Thai       : -{mask_thai.sum()} baris')

# Teks yang hanya berisi tanggal (artefak merge)
mask_only_date = df_clean['teks_bersih'].str.match(
    r'^\s*(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d+\s*$', na=False
)
df_clean = df_clean[~mask_only_date]
print(f'Drop teks hanya tanggal: -{mask_only_date.sum()} baris')

# Akun media/portal berita & bot otomatis (bukan keluhan warga)
pola_media = (
    r'news|post|tv|radio|detik|kompas|tribun|antara|tempo|'
    r'kumparan|idntimes|liputan|berita|times|journal|pos|'
    r'radar|bisnis|metro|elshinta|bmkg|lines|kanal|merdeka|'
    r'sindonews|pikiran|rakyat|suara|okezone|grid|viva'
)
mask_media = df_clean['username'].str.lower().str.contains(pola_media, regex=True, na=False)
df_clean = df_clean[~mask_media]
print(f'Drop akun media/bot    : -{mask_media.sum()} baris')

# Teks berkarakter Arab / Korea / Jepang
mask_nonlatin = df_clean['teks_bersih'].str.contains(
    r'[\u0600-\u06FF\u3040-\u30FF\uAC00-\uD7A3]', regex=True, na=False
)
df_clean = df_clean[~mask_nonlatin]
print(f'Drop teks non-latin    : -{mask_nonlatin.sum()} baris')

# Duplikat teks
n_sebelum_dup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['teks_bersih'], keep='first')
print(f'Drop duplikat teks     : -{n_sebelum_dup - len(df_clean)} baris')

df_clean = df_clean.reset_index(drop=True)
print(f'\nSisa setelah drop      : {len(df_clean)} baris (dari {n_awal})')


Drop bahasa Thai       : -44 baris
Drop teks hanya tanggal: -12 baris
Drop akun media/bot    : -92 baris
Drop teks non-latin    : -31 baris
Drop duplikat teks     : -54 baris

Sisa setelah drop      : 1585 baris (dari 1818)


5.2. Bersihkan Artefak yang Tersisa di Teks

In [6]:
def bersihkan_artefak(teks):
    if not isinstance(teks, str):
        return teks
    # Hapus artefak waktu relatif di akhir teks, contoh: 'banjir 8m' → 'banjir'
    teks = re.sub(r'\s+\d+[mhd]\s*$', '', teks)
    # Hapus angka desimal/engagement di akhir, contoh: 'rusak 1.0' → 'rusak'
    teks = re.sub(r'\s+\d+\.\d+K?\s*$', '', teks)
    # Hapus URL
    teks = re.sub(r'https?://\S*', '', teks)
    # Hapus simbol '#' tapi pertahankan katanya, contoh: '#Banjir' → 'Banjir'
    teks = re.sub(r'#(\w+)', r'\1', teks)
    # Rapikan spasi
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

df_clean['teks_bersih'] = df_clean['teks_bersih'].apply(bersihkan_artefak)

# Drop baris yang teksnya jadi terlalu pendek setelah dibersihkan
n_sebelum = len(df_clean)
df_clean = df_clean[df_clean['teks_bersih'].str.len() >= 15]
df_clean = df_clean.reset_index(drop=True)

print(f'Drop teks <15 karakter (post-clean): -{n_sebelum - len(df_clean)} baris')
print(f'Sisa setelah bersihkan artefak     : {len(df_clean)} baris')
display(df_clean[['waktu', 'username', 'teks_bersih']].head(3))


Drop teks <15 karakter (post-clean): -40 baris
Sisa setelah bersihkan artefak     : 1545 baris


,waktu,username,teks_bersih
0,8m,@alung_koh21471,Koh Alung krisis air Pemerintah berupaya menan...
1,17m,@ciroalves2055,Rizki Fadillnaldo KRISIS AIR BERSIH? JANGAN AS...
2,1h,@indrakusuma,Indra Kusuma air keruh Mangkanya doi dkk kecew...


5.3. Normalisasi & Imputasi Kolom Waktu

In [7]:
# CELL 6c: Normalisasi & Imputasi Kolom Tanggal (Sesuai Format Kaggle)
import re
from datetime import datetime, timedelta

TANGGAL_SCRAPING = pd.Timestamp('2026-03-26')

def parse_waktu(nilai):
    if pd.isna(nilai) or not isinstance(nilai, str):
        return pd.NaT
    nilai = nilai.strip()
    
    # 1. Format relatif: '8m', '3h', '2d'
    cocok_relatif = re.match(r'^(\d+)([mhd])$', nilai)
    if cocok_relatif:
        angka, satuan = int(cocok_relatif.group(1)), cocok_relatif.group(2)
        delta = {'m': timedelta(minutes=angka), 'h': timedelta(hours=angka), 'd': timedelta(days=angka)}
        return TANGGAL_SCRAPING - delta[satuan]
        
    # 2. Format absolut: 'Apr 3'
    try:
        return pd.Timestamp(datetime.strptime(nilai, '%b %d').replace(year=TANGGAL_SCRAPING.year))
    except ValueError:
        return pd.NaT

# Eksekusi transformasi awal
df_clean['tanggal'] = df_clean['waktu'].apply(parse_waktu)

# KONVERSI KE STRING (Agar sama dengan format temanmu: YYYY-MM-DD)
df_clean['tanggal'] = pd.to_datetime(df_clean['tanggal']).dt.strftime('%Y-%m-%d')

# Imputasi NaT dengan String tanggal default
n_nat = df_clean['tanggal'].isna().sum()
df_clean['tanggal'] = df_clean['tanggal'].fillna('2026-03-26')

# Drop kolom original waktu
df_clean = df_clean.drop(columns=['waktu'])

print(f'✅ Diimputasi (26 Mar 26) : {n_nat} baris')
print('✅ Format kolom "tanggal" sekarang adalah String (YYYY-MM-DD)')
print('\nDistribusi tanggal:')
print(df_clean['tanggal'].value_counts().sort_index())

✅ Diimputasi (26 Mar 26) : 469 baris
✅ Format kolom "tanggal" sekarang adalah String (YYYY-MM-DD)

Distribusi tanggal:
tanggal
2026-03-25    568
2026-03-26    469
2026-03-27     13
2026-03-28     81
2026-03-29     65
2026-03-30     74
2026-03-31     52
2026-04-01     51
2026-04-02     67
2026-04-03    105
Name: count, dtype: int64


5.4. Ringkasan Akhir & Ekspor Data Bersih

In [8]:
# Susun urutan kolom final (tepat 8 kolom, tanpa waktu & tanggal_valid)
kolom_final = ['sumber', 'tanggal', 'username', 'teks_bersih', 
            'kategori_isu', 'urgensi', 'sentimen', 'lokasi_insiden']

if 'sumber' not in df_clean.columns:
    df_clean.insert(0, 'sumber', 'twitter')
    
df_output = df_clean[kolom_final].copy()

# Ekspor
output_path = '../data/processed/twitter_keluhan_clean.csv'
try:
    df_output.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'✅ Sukses! Data bersih diekspor ke: {output_path}')
except Exception as e:
    print(f'❌ Gagal menyimpan file: {e}')

# Ringkasan
dist = df_output['kategori_isu'].value_counts()
print(f'\nShape output   : {df_output.shape[0]} baris x {df_output.shape[1]} kolom')
print(f'Raw awal       : 1873 baris  →  Clean: {df_output.shape[0]} baris '
        f'({df_output.shape[0]/1873*100:.1f}% dipertahankan)')
print(f'Imbalance ratio: {dist.max()}/{dist.min()} = {dist.max()/dist.min():.1f}x')
print('\n--- Distribusi Kategori Akhir ---')
print(dist)
print('\nPreview Data Final:')
display(df_output.head(3))

✅ Sukses! Data bersih diekspor ke: ../data/processed/twitter_keluhan_clean.csv

Shape output   : 1545 baris x 8 kolom
Raw awal       : 1873 baris  →  Clean: 1545 baris (82.5% dipertahankan)
Imbalance ratio: 354/37 = 9.6x

--- Distribusi Kategori Akhir ---
kategori_isu
Infrastruktur & Jalan    354
Ekonomi & UMKM           290
Lingkungan & Sampah      161
Pelayanan Publik         151
Pendidikan               149
Air & Sanitasi           134
Transportasi              98
Keamanan & Sosial         97
Bencana & Cuaca           74
Kesehatan                 37
Name: count, dtype: int64

Preview Data Final:


,sumber,tanggal,username,teks_bersih,kategori_isu,urgensi,sentimen,lokasi_insiden
0,twitter,2026-03-25,@alung_koh21471,Koh Alung krisis air Pemerintah berupaya menan...,Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1,twitter,2026-03-25,@ciroalves2055,Rizki Fadillnaldo KRISIS AIR BERSIH? JANGAN AS...,Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
2,twitter,2026-03-25,@indrakusuma,Indra Kusuma air keruh Mangkanya doi dkk kecew...,Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
